# Windows Repository Bridge — direct Colab vision server

The desktop app sends screenshots directly to this temporary authenticated HTTPS endpoint. Colab returns the analysis to the app, which publishes the result through the normal GitHub mailbox tool result. The notebook does not read or write GitHub, and **does not need a GitHub token or Colab Secret**.

1. Open this notebook while the Windows Repository Bridge app is running.
2. Connect a Colab runtime (choose a GPU runtime if available) and run the worker cell.
3. The cell creates a temporary Cloudflare Quick Tunnel and automatically registers the tunnel with the app through a one-time localhost browser handshake. No URL, token, or key needs to be copied by hand.
4. Connecting does not load the model. In the existing ChatGPT conversation, call `vision_activate` only when visual analysis is needed. The first activation may download about 6.2 GB and compile llama.cpp; Google Drive is used to cache weights between runtimes.

The endpoint is limited to a random in-memory bearer key, accepts only the documented health/activation/deactivation/screenshot-analysis routes, and checks each screenshot SHA-256. The model/API is bound to localhost inside Colab; only the authenticated endpoint is tunneled. Deactivation leaves the Colab runtime and model running. Colab GPU access and session uptime are not guaranteed.

In [ ]:
# @title Start the direct Colab vision server
import base64
import hashlib
import hmac
import importlib.util
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests
from google.colab import drive
from IPython.display import Javascript, display

MODEL_REPO = "Qwen/Qwen3-VL-8B-Instruct-GGUF"
MODEL_FILE = "Qwen3VL-8B-Instruct-Q4_K_M.gguf"
PROJECTOR_FILE = "mmproj-Qwen3VL-8B-Instruct-F16.gguf"
MODEL_URL = "https://huggingface.co/" + MODEL_REPO + "/resolve/main/" + MODEL_FILE
PROJECTOR_URL = "https://huggingface.co/" + MODEL_REPO + "/resolve/main/" + PROJECTOR_FILE
SERVER_URL = "http://127.0.0.1:8080"
DIRECT_API_PORT = 8765
DIRECT_PROTOCOL = "chatgpt-colab-vision-direct/v1"
RESULT_PROTOCOL = "chatgpt-colab-vision-result/v1"
MAX_SCREENSHOT_BYTES = 600 * 1024
POLL_SECONDS = 3
HEARTBEAT_SECONDS = 15
API_KEY = secrets.token_urlsafe(32)

_server = None
_tunnel_process = None
_tunnel_log_handle = None
_model_loaded = False
_enabled = False
_gpu = None
_vram_mib = None
_active_request = None
_last_completed = None
_runtime_lock = threading.RLock()
_inference_lock = threading.Lock()
_last_setup_attempt = 0.0
_runtime_state = {
    "status": "inactive",
    "stage": "server_start",
    "message": "Direct Colab service is starting. Vision remains disabled until the app calls vision_activate.",
    "model_loaded": False,
    "gpu": None,
    "vram_mib": None,
    "active_request_id": None,
    "last_completed_request_id": None,
    "updated_at": None
}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def publish_runtime(status, stage=None, message=None, **extra):
    global _runtime_state
    with _runtime_lock:
        payload = dict(_runtime_state)
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "status": status,
            "enabled": bool(_enabled),
            "heartbeat_at": utc_now(),
            "updated_at": utc_now(),
            "worker": "colab-direct-vision/v1",
            "model_repository": MODEL_REPO,
            "model": MODEL_FILE,
            "projector": PROJECTOR_FILE,
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "stage": stage if stage is not None else payload.get("stage"),
            "message": (message if message is not None else payload.get("message") or "")[:1000]
        })
        payload.update(extra)
        _runtime_state = payload
        return dict(payload)

def check_gpu():
    if not shutil.which("nvidia-smi"):
        return None
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=15
    )
    line = next((line.strip() for line in result.stdout.splitlines() if line.strip()), "")
    columns = [column.strip() for column in line.split(",")]
    if len(columns) < 3:
        raise RuntimeError("Could not parse nvidia-smi GPU/VRAM output.")
    return columns[0], int(columns[1]), int(columns[2])

def ensure_server():
    global _server, _model_loaded, _gpu, _vram_mib, _last_setup_attempt
    if _server is not None and _server.poll() is None:
        try:
            if requests.get(SERVER_URL + "/health", timeout=2).ok:
                _model_loaded = True
                publish_runtime("ready", stage="ready", message="The vision model is already loaded.")
                return True
        except requests.RequestException:
            pass
    if time.monotonic() - _last_setup_attempt < 45:
        return False
    _last_setup_attempt = time.monotonic()
    publish_runtime("starting", stage="drive_mount", message="Preparing persistent model storage. Google Drive authorization may be requested once.")
    try:
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        publish_runtime("needs_authentication", stage="drive_mount", message="Google Drive authorization is required in the Colab browser: " + str(exc))
        return False

    gpu = check_gpu()
    if not gpu:
        publish_runtime("gpu_unavailable", stage="detect_gpu", message="No NVIDIA GPU is attached to this Colab runtime. Select a GPU runtime if one is available.")
        return False
    _gpu, total_mib, free_mib = gpu
    _vram_mib = total_mib
    if total_mib < 10000:
        publish_runtime("gpu_unavailable", stage="check_vram", message=f"Detected {_gpu} with {total_mib} MiB VRAM. This 8B model and projector may not fit.")
        return False
    if not shutil.which("nvcc"):
        publish_runtime("error", stage="check_cuda", message="An NVIDIA GPU is present but nvcc is missing; the CUDA llama.cpp build cannot be verified.")
        return False

    root = Path("/content/drive/MyDrive/WindowsRepositoryBridgeVision")
    model_dir = root / "models"
    hf_cache = root / "hf-cache"
    model_dir.mkdir(parents=True, exist_ok=True)
    hf_cache.mkdir(parents=True, exist_ok=True)
    needed = [model_dir / MODEL_FILE, model_dir / PROJECTOR_FILE]
    free_bytes = shutil.disk_usage(root).free
    if any(not item.exists() for item in needed) and free_bytes < 7_000_000_000:
        publish_runtime("error", stage="check_drive_space", message=f"Persistent Drive storage has only {free_bytes // (1024**3)} GiB free; at least 7 GB is required.")
        return False

    publish_runtime("starting", stage="dependencies", message="Checking one-time model and build dependencies.")
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True, timeout=300)
    from huggingface_hub import hf_hub_download

    publish_runtime("starting", stage="download_model", message="Reusing or downloading the Q4_K_M model and matching F16 vision projector.")
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=str(model_dir), cache_dir=str(hf_cache))
    projector_path = hf_hub_download(repo_id=MODEL_REPO, filename=PROJECTOR_FILE, local_dir=str(model_dir), cache_dir=str(hf_cache))
    if Path(model_path).stat().st_size < 4_500_000_000 or Path(projector_path).stat().st_size < 1_000_000_000:
        raise RuntimeError("Model or projector file is unexpectedly small; refusing to use a partial download.")

    llama_source = Path("/content/llama.cpp")
    if not (llama_source / "CMakeLists.txt").exists():
        publish_runtime("starting", stage="clone_llama_cpp", message="Fetching upstream llama.cpp source.")
        subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp.git", str(llama_source)],
                       check=True, timeout=600)
    build_dir = llama_source / "build"
    server_binary = build_dir / "bin" / "llama-server"
    if not server_binary.exists():
        if not shutil.which("cmake") or not shutil.which("make"):
            subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=300)
            subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake", "git"],
                           check=True, timeout=600)
        publish_runtime("starting", stage="build_llama_cpp", message="Building llama-server with CUDA and multimodal support.")
        subprocess.run(["cmake", "-S", str(llama_source), "-B", str(build_dir),
                        "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF"],
                       check=True, timeout=900)
        subprocess.run(["cmake", "--build", str(build_dir), "--config", "Release",
                        "--target", "llama-server", "-j", "2"],
                       check=True, timeout=1800)
    if not server_binary.exists():
        raise RuntimeError("The llama.cpp build ended without a llama-server executable.")
    help_text = subprocess.run([str(server_binary), "--help"], check=True, capture_output=True, text=True, timeout=30).stdout
    for required_flag in ("--mmproj", "--host", "--port", "--ctx-size", "--n-gpu-layers"):
        if required_flag not in help_text:
            raise RuntimeError("The built llama-server does not expose required CLI flag: " + required_flag)

    if _server is not None and _server.poll() is None:
        _server.terminate()
        try:
            _server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            _server.kill()
    log_path = Path("/content/llama-server-vision.log")
    log_handle = open(log_path, "a", encoding="utf-8")
    _server = subprocess.Popen([
        str(server_binary), "-m", str(model_path), "--mmproj", str(projector_path),
        "-c", "8192", "--n-gpu-layers", "99", "--host", "127.0.0.1", "--port", "8080"
    ], stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    publish_runtime("starting", stage="load_model", message="llama-server started; waiting for model health.")
    deadline = time.monotonic() + 240
    while time.monotonic() < deadline:
        if _server.poll() is not None:
            try:
                tail = log_path.read_text(encoding="utf-8", errors="replace")[-2500:]
            except Exception:
                tail = "(server log unavailable)"
            raise RuntimeError("llama-server exited before reporting healthy. Diagnostic tail: " + tail)
        try:
            if requests.get(SERVER_URL + "/health", timeout=3).ok:
                _model_loaded = True
                publish_runtime("ready", stage="ready", message="Model and vision projector are loaded and llama-server health succeeded.")
                log_handle.close()
                return True
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("Timed out waiting for llama-server /health after loading the Qwen3-VL model.")

class DirectVisionHandler(BaseHTTPRequestHandler):
    server_version = "BridgeVision/1"

    def log_message(self, fmt, *args):
        # Do not log bearer credentials, image payloads, or user prompts.
        return

    def _authorized(self):
        value = self.headers.get("Authorization", "")
        return hmac.compare_digest(value, "Bearer " + API_KEY)

    def _send(self, status, payload):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Cache-Control", "no-store")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        if self.path != "/health":
            self._send(404, {"error": "Not found"})
            return
        with _runtime_lock:
            payload = dict(_runtime_state)
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "enabled": bool(_enabled),
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "heartbeat_at": utc_now(),
            "updated_at": utc_now()
        })
        self._send(200, payload)

    def do_POST(self):
        global _enabled, _active_request, _last_completed
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        try:
            length = int(self.headers.get("Content-Length", "0"))
        except ValueError:
            self._send(400, {"error": "Invalid Content-Length"})
            return
        if length < 1 or length > 2_000_000:
            self._send(413, {"error": "Request body must be between 1 byte and 2 MB"})
            return
        try:
            body = json.loads(self.rfile.read(length).decode("utf-8"))
        except Exception:
            self._send(400, {"error": "Malformed JSON"})
            return
        if not isinstance(body, dict) or body.get("protocol") != DIRECT_PROTOCOL:
            self._send(400, {"error": "Protocol mismatch"})
            return
        if self.path == "/activate":
            _enabled = True
            publish_runtime("starting", stage="activation", message="Vision activation received. Model setup is starting.")
            self._send(202, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": True})
            return
        if self.path == "/deactivate":
            _enabled = False
            publish_runtime("inactive", stage="idle", message="Vision is disabled; the runtime and any loaded model remain running.")
            self._send(200, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": False, "runtime_kept_alive": True})
            return
        if self.path != "/analyze":
            self._send(404, {"error": "Not found"})
            return
        if not _enabled:
            self._send(409, {"error": "Vision is inactive; call vision_activate first."})
            return
        if not _model_loaded:
            self._send(409, {"error": "Vision model is not loaded yet; check vision_status and retry the same request ID."})
            return
        with _inference_lock:
            try:
                job_id = body.get("request_id")
                if not isinstance(job_id, str) or not re.fullmatch(r"vision-[a-f0-9]{24}", job_id):
                    self._send(400, {"error": "Malformed request ID"})
                    return
                expected_hash = body.get("sha256")
                image_b64 = body.get("image_base64")
                if not isinstance(expected_hash, str) or not re.fullmatch(r"[a-f0-9]{64}", expected_hash):
                    self._send(400, {"error": "Malformed SHA-256"})
                    return
                if not isinstance(image_b64, str):
                    self._send(400, {"error": "Image payload is missing"})
                    return
                image_bytes = base64.b64decode(image_b64, validate=True)
                if not image_bytes or len(image_bytes) > MAX_SCREENSHOT_BYTES:
                    self._send(413, {"error": "Screenshot is empty or larger than the 600 KiB limit"})
                    return
                actual_hash = hashlib.sha256(image_bytes).hexdigest()
                if not hmac.compare_digest(actual_hash, expected_hash):
                    self._send(400, {"error": "Screenshot SHA-256 did not match the submitted value"})
                    return
                _active_request = job_id
                publish_runtime("processing", stage="inference", message="Analyzing one screenshot received from the desktop app.")
                started = time.monotonic()
                request_payload = {
                    "model": "qwen3-vl-8b-vision",
                    "temperature": 0.1,
                    "max_tokens": 1800,
                    "stream": False,
                    "messages": [
                        {"role": "system", "content": (
                            "You are a careful visual observer, not a general reasoning agent. "
                            "Describe only details supported by the provided image. Do not infer unseen state. "
                            "Mark illegible text and uncertainty explicitly. Organize the report into: Overview; "
                            "Visible objects, panels, controls and layout; Text transcription; Visible errors or anomalies; "
                            "Evidence supporting observations; Uncertainties."
                        )},
                        {"role": "user", "content": [
                            {"type": "text", "text": str(body.get("prompt") or "")[:2000]},
                            {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + image_b64}}
                        ]}
                    ]
                }
                try:
                    response = requests.post(SERVER_URL + "/v1/chat/completions", json=request_payload, timeout=(10, 300))
                    if not response.ok:
                        raise RuntimeError("llama-server HTTP " + str(response.status_code) + ": " + response.text[:600])
                    output = response.json()
                    choices = output.get("choices")
                    if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
                        raise RuntimeError("llama-server returned a malformed completion payload.")
                    message = choices[0].get("message", {})
                    report = message.get("content") if isinstance(message, dict) else None
                    if isinstance(report, list):
                        report = "\n".join(str(part.get("text", "")) for part in report if isinstance(part, dict))
                    if not isinstance(report, str) or not report.strip():
                        raise RuntimeError("The model returned an empty or non-text report.")
                    report = report.strip()
                    if len(report) > 30000:
                        report = report[:30000] + "\n\n[Report truncated at the safety limit.]"
                    _last_completed = job_id
                    result = {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "completed",
                        "completed_at": utc_now(),
                        "model_repository": MODEL_REPO,
                        "model": MODEL_FILE,
                        "input_sha256": actual_hash,
                        "inference_ms": int((time.monotonic() - started) * 1000),
                        "report": report
                    }
                    self._send(200, result)
                except Exception as exc:
                    self._send(200, {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "failed",
                        "completed_at": utc_now(),
                        "input_sha256": actual_hash,
                        "error": type(exc).__name__ + ": " + str(exc)[:1200]
                    })
                finally:
                    _active_request = None
                    publish_runtime("ready" if _model_loaded else "error", stage="ready" if _model_loaded else "inference_error",
                                    message="Model is ready for another request." if _model_loaded else "Model health was lost during inference.")
            except Exception as exc:
                self._send(400, {"error": type(exc).__name__ + ": " + str(exc)[:800]})

def api_server_loop():
    server = ThreadingHTTPServer(("127.0.0.1", DIRECT_API_PORT), DirectVisionHandler)
    server.daemon_threads = True
    print("Authenticated vision API listening on localhost port", DIRECT_API_PORT)
    server.serve_forever()

def worker_loop():
    while True:
        try:
            if _enabled:
                if not _model_loaded:
                    try:
                        ensure_server()
                    except Exception as exc:
                        publish_runtime("error", stage="setup_error", message=type(exc).__name__ + ": " + str(exc))
                else:
                    publish_runtime("ready", stage="ready", message="Vision is enabled and the model is ready.")
            else:
                publish_runtime("inactive", stage="idle", message="Direct Colab server is connected; vision is disabled and model setup will not start.")
        except Exception as exc:
            publish_runtime("error", stage="worker_error", message=type(exc).__name__ + ": " + str(exc))
        time.sleep(POLL_SECONDS)

def ensure_cloudflared():
    binary = shutil.which("cloudflared")
    if binary:
        return binary
    binary_path = Path("/content/cloudflared")
    if not binary_path.exists():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        binary_path.write_bytes(response.content)
        binary_path.chmod(0o755)
    if not os.access(binary_path, os.X_OK):
        binary_path.chmod(0o755)
    return str(binary_path)

def start_cloudflared():
    global _tunnel_process, _tunnel_log_handle
    binary = ensure_cloudflared()
    log_path = Path("/content/bridge-cloudflared.log")
    _tunnel_log_handle = open(log_path, "w", encoding="utf-8")
    _tunnel_process = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:" + str(DIRECT_API_PORT)],
        stdout=_tunnel_log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
    deadline = time.monotonic() + 90
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    while time.monotonic() < deadline:
        if _tunnel_process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-1600:]
            raise RuntimeError("cloudflared exited before creating its HTTPS tunnel: " + tail)
        text = log_path.read_text(encoding="utf-8", errors="replace")
        match = pattern.search(text)
        if match:
            return match.group(0)
        time.sleep(1)
    raise RuntimeError("Timed out waiting for a Cloudflare Quick Tunnel URL. Check /content/bridge-cloudflared.log.")

def register_tunnel_with_desktop(tunnel_url):
    payload_json = json.dumps({
        "protocol": DIRECT_PROTOCOL,
        "tunnel_url": tunnel_url,
        "api_key": API_KEY
    })
    js = """
(async function() {
  const payload = __PAYLOAD__;
  const ports = Array.from({length: 12}, (_, index) => 17843 + index);
  let lastError = "No local app registration listener was found.";
  for (const port of ports) {
    try {
      const base = "http://127.0.0.1:" + port;
      const challengeResponse = await fetch(base + "/vision/handshake", {method:"GET", mode:"cors", cache:"no-store"});
      if (!challengeResponse.ok) continue;
      const challenge = await challengeResponse.json();
      if (challenge.protocol !== payload.protocol || typeof challenge.nonce !== "string") continue;
      const response = await fetch(base + "/vision/register", {
        method:"POST", mode:"cors", cache:"no-store",
        headers:{"Content-Type":"application/json"},
        body:JSON.stringify({...payload, nonce:challenge.nonce})
      });
      const result = await response.json();
      if (response.ok && result.ok === true) {
        console.info("Windows Repository Bridge: direct Colab tunnel registered.");
        return;
      }
      lastError = result.error || ("Registration failed with HTTP " + response.status);
    } catch (error) {
      lastError = String(error && error.message || error);
    }
  }
  console.warn("Windows Repository Bridge auto-registration failed. Keep the app open, confirm Colab is allowed to reach localhost, and rerun this cell. Detail: " + lastError);
})();
""".replace("__PAYLOAD__", payload_json)
    display(Javascript(js))

print("Starting direct Colab vision server. No GitHub token or Colab Secret is needed.")
publish_runtime("starting", stage="api_server", message="Starting the authenticated direct HTTPS vision endpoint.")
threading.Thread(target=api_server_loop, name="bridge-direct-api", daemon=True).start()
time.sleep(1)
tunnel_url = start_cloudflared()
print("Secure tunnel created. Registering it with the running Windows Repository Bridge app...")
register_tunnel_with_desktop(tunnel_url)
print("Connection registration was sent to the browser. The app should report Colab connected shortly.")
threading.Thread(target=worker_loop, name="bridge-vision-worker", daemon=True).start()
print("Vision is still disabled. Use vision_activate in the ChatGPT bridge tools when visual analysis is needed.")
print("Keep this cell running. Colab GPU availability and runtime uptime are not guaranteed.")
while True:
    time.sleep(60)
